# Game Team Stats and Game Player Stats

Build one row per team per game and one row per player per game from `game_processed_events`.

In [16]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from pymongo import MongoClient, UpdateOne

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)



In [17]:
def load_config(path: str | Path = "../config/config.yaml") -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        path = Path("config/config.yaml")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


config = load_config()
mongo_config = config["mongo"]
season = config["season"]["year"]
processed_collection_name = mongo_config["collection"]["collection_processed_events"]
team_stats_collection_name = mongo_config["collection"].get("collection_game_team_stats", "game_team_stats")
player_stats_collection_name = mongo_config["collection"].get("collection_game_player_stats", "game_player_stats")

client = MongoClient(mongo_config["url"])
db = client[mongo_config["db"]]
processed_collection = db[processed_collection_name]
team_stats_collection = db[team_stats_collection_name]
player_stats_collection = db[player_stats_collection_name]

print(f"DB: {db.name}")
print(f"Processed events collection: {processed_collection_name}")
print(f"Team stats collection: {team_stats_collection_name}")
print(f"Player stats collection: {player_stats_collection_name}")



DB: WhoScored
Processed events collection: game_processed_events
Team stats collection: game_team_stats
Player stats collection: game_player_stats


## Load Processed Events

In [18]:
GAME_ID = 1910873  # change this when you want to inspect a specific game

def load_game_events(game_id: int) -> pd.DataFrame:
    return pd.DataFrame(
        list(processed_collection.find({"season": season, "game_id": int(game_id)}, {"_id": 0}))
    )


def prepare_game_events(events: pd.DataFrame) -> pd.DataFrame:
    if events.empty:
        return events

    events = events.copy()

    home_team_id = events["home_team_id"].iloc[0]
    away_team_id = events["away_team_id"].iloc[0]
    home_team_name = events["home_team_name"].iloc[0]
    away_team_name = events["away_team_name"].iloc[0]

    venue_map = {
        home_team_id: "home",
        away_team_id: "away",
    }

    team_name_map = {
        home_team_id: home_team_name,
        away_team_id: away_team_name,
    }

    events["game_venue"] = events["team_id"].map(venue_map).fillna("neutral")
    events["team_name"] = events["team_id"].map(team_name_map).fillna(events["team"])

    events["opponent_team_id"] = events["team_id"].map({
        home_team_id: away_team_id,
        away_team_id: home_team_id,
    })

    events["opponent_team_name"] = events["team_id"].map({
        home_team_id: away_team_name,
        away_team_id: home_team_name,
    })

    foul_mask = events["type"].eq("Foul")
    foul_committed = foul_mask & events["outcome_type"].eq("Unsuccessful")
    foul_suffered = foul_mask & events["outcome_type"].eq("Successful")
    events.loc[foul_mask, "foul_committed"] = foul_committed[foul_mask]
    events.loc[foul_mask, "foul_suffered"] = foul_suffered[foul_mask]
    if "foul_type" in events.columns:
        events.loc[foul_committed, "foul_type"] = "Foul Committed"
        events.loc[foul_suffered, "foul_type"] = "Foul Suffered"

    return events



sample_game_events = prepare_game_events(load_game_events(GAME_ID))
sample_game_events


,game_id,season,competition_country,competition_name,game_date,game_status,week,home_team_id,home_team_name,away_team_id,away_team_name,event_idx,period,minute,second,expanded_minute,team_id,team,player_id,player,type,outcome_type,x,y,end_x,end_y,goal_mouth_y,goal_mouth_z,blocked_x,blocked_y,related_event_id,related_player_id,stat_event_type,shot_event,pass_event,pass_completed,dribble_event,tackle_attempted_event,interception_event,clearance_event,block_event,offside_event,foul_event,aerial_duel_event,touch_event,loss_possession_event,error_event,save_event,claim_event,punch_event,goalkeeper_event,ball_recovery_event,card_event,substitution_event,shot_goal,shot_on_target,shot_off_target,shot_woodwork,shot_blocked,shot_own_goal,shot_zone_6_yard_box,shot_zone_penalty_area,shot_zone_outside_box,shot_open_play,shot_fastbreak,shot_set_piece,shot_penalty,shot_right_foot,shot_left_foot,shot_head,shot_other_body_part,shot_result,shot_zone,shot_situation,shot_body_part,pass_attempt,pass_incomplete,pass_cross,pass_freekick,pass_corner,pass_through_ball,pass_throw_in,pass_key_pass,pass_key_pass_qualifier,pass_long,pass_short,pass_length,pass_chipped,pass_ground,pass_height,pass_head,pass_feet,pass_body_part,pass_forward,pass_backward,pass_left,pass_right,pass_defensive_third,pass_mid_third,pass_final_third,pass_target_zone,dribble_successful,dribble_unsuccessful,tackle_gained_possession,tackle_did_not_get_possession,tackle_was_dribbled,tackle_result,clearance_head,clearance_feet,clearance_body_part,blocked_shot,blocked_cross,block_type,caught_offside,offside_pass,offside_provoked,offside_type,foul_committed,foul_suffered,foul_type,aerial_duel_won,aerial_duel_lost,aerial_result,dispossessed,turnover,loss_possession_type,error_leading_to_shot,error_leading_to_goal,error_type,keeper_pickup,keeper_sweeper,gk_type,yellow_card,second_yellow_card,red_card,card_type,substitution_on,substitution_off,substitution_type,game_venue,team_name,opponent_team_id,opponent_team_name
0,1910873,2025-2026,Germany,Bundesliga,2026-04-24 19:30:00,finished,31,7614,RB Leipzig,796,Union Berlin,2,FirstHalf,0,0.0,0,796,Union Berlin,369687.0,András Schäfer,Pass,Successful,50.0,50.0,34.6,59.7,NaN,NaN,NaN,NaN,NaN,NaN,pass,False,True,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,NaN,NaN,True,False,False,False,False,False,False,False,False,False,True,Short,False,True,Ground,False,True,Feet,False,True,True,False,False,True,False,Mid Third,False,False,False,False,False,NaN,False,False,NaN,False,False,NaN,False,False,False,NaN,False,False,NaN,False,False,NaN,False,False,NaN,False,False,NaN,False,False,NaN,False,False,False,NaN,False,False,NaN,away,Union Berlin,7614,RB Leipzig
1,1910873,2025-2026,Germany,Bundesliga,2026-04-24 19:30:00,finished,31,7614,RB Leipzig,796,Union Berlin,3,FirstHalf,0,2.0,0,796,Union Berlin,303076.0,Danilho Doekhi,Pass,Successful,34.6,59.7,73.6,74.4,NaN,NaN,NaN,NaN,NaN,NaN,pass,False,True,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,NaN,NaN,True,False,False,False,False,False,False,False,False,True,False,Long,False,True,Ground,False,True,Feet,True,False,True,False,False,False,True,Final Third,False,False,False,False,False,NaN,False,False,NaN,False,False,NaN,False,False,False,NaN,False,False,NaN,False,False,NaN,False,False,NaN,False,False,NaN,False,False,NaN,False,False,False,NaN,False,False,NaN,away,Union Berlin,7614,RB Leipzig
2,1910873,2025-2026,Germany,Bundesliga,2026-04-24 19:30:00,finished,31,7614,RB Leipzig,796,Union Berlin,4,FirstHalf,0,5.0,0,796,Union Berlin,426239.0,Andrej Ilic,Aerial,Successful,73.6,74.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,aerial_duel,False,False,False,False,False,False,False,False,

## Stat Columns

In [19]:
STAT_COLUMNS = config.get("game_stats", {}).get("team_stats", [])

In [20]:

stat_columns = [c for c in STAT_COLUMNS if c in sample_game_events.columns]
missing_stat_columns = sorted(set(STAT_COLUMNS) - set(stat_columns))
print(f"Using {len(stat_columns)} stat columns")
print(f"Missing from current processed data: {missing_stat_columns}")

sample_game_events[stat_columns] = sample_game_events[stat_columns].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)



Using 77 stat columns
Missing from current processed data: []


In [21]:
def safe_div(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)


def add_rate_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if {"shot_zone_6_yard_box", "shot_zone_penalty_area"}.issubset(df.columns):
        df["shot_zone_inside_box"] = df["shot_zone_6_yard_box"] + df["shot_zone_penalty_area"]
    if {"shot_right_foot", "shot_left_foot"}.issubset(df.columns):
        df["shot_foot"] = df["shot_right_foot"] + df["shot_left_foot"]


    PERC_STATS = config.get("game_stats", {}).get("perc_stats", {})

    for new_col, formula in PERC_STATS.items():
        numerator = formula["numerator"]
        denominator = formula["denominator"]

        df[new_col] = safe_div(
            df[numerator],
            df[denominator],
        )
    return df


## Build Game Team Stats

In [22]:
config.get("game_stats", {})



{'team_match_features': ['game_id',
  'game_date',
  'game_status',
  'competition_name',
  'competition_country',
  'season',
  'week',
  'team_id',
  'team_name',
  'opponent_team_id',
  'opponent_team_name',
  'game_venue'],
 'player_match_features': ['game_id',
  'game_date',
  'game_status',
  'competition_name',
  'competition_country',
  'season',
  'week',
  'team_id',
  'team_name',
  'opponent_team_id',
  'opponent_team_name',
  'game_venue',
  'player_id',
  'player',
  'starting_lineup',
  'minutes_played'],
 'team_stats': ['shot_event',
  'shot_goal',
  'shot_on_target',
  'shot_off_target',
  'shot_woodwork',
  'shot_blocked',
  'shot_own_goal',
  'shot_zone_6_yard_box',
  'shot_zone_penalty_area',
  'shot_zone_outside_box',
  'shot_open_play',
  'shot_fastbreak',
  'shot_set_piece',
  'shot_penalty',
  'shot_right_foot',
  'shot_left_foot',
  'shot_head',
  'shot_other_body_part',
  'pass_attempt',
  'pass_completed',
  'pass_cross',
  'pass_freekick',
  'pass_corner',
 

In [23]:

GAME_TEAM_FEATURES = config.get("game_stats", {}).get("team_match_features", [])

PASS_COMPLETION_BASE_COLUMNS = config.get("game_stats", {}).get("pass_complettion_stats", [])


def add_derived_event_stats(game_events: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    game_events = game_events.copy()
    derived_columns: list[str] = []
    if "pass_completed" in game_events.columns:
        pass_completed = pd.to_numeric(game_events["pass_completed"], errors="coerce").fillna(0).astype(int)
        for col in PASS_COMPLETION_BASE_COLUMNS:
            if col in game_events.columns:
                completed_col = f"{col}_completed"
                base = pd.to_numeric(game_events[col], errors="coerce").fillna(0).astype(int)
                game_events[completed_col] = (base.eq(1) & pass_completed.eq(1)).astype(int)
                derived_columns.append(completed_col)
    return game_events, derived_columns


def interval_masks(game_events: pd.DataFrame) -> dict[str, pd.Series]:
    minute = pd.to_numeric(game_events["minute"], errors="coerce")
    period = game_events["period"].fillna("")
    first_half = period.eq("FirstHalf")
    second_half = period.eq("SecondHalf")
    return {
        "ft": pd.Series(True, index=game_events.index),
        "fh": first_half,
        "sh": second_half,
        "m_1_15": minute.lt(15),
        "m_16_30": minute.ge(15) & minute.lt(30),
        "m_31_45": first_half & minute.ge(30),
        "m_46_60": second_half & minute.lt(60),
        "m_61_75": second_half & minute.ge(60) & minute.lt(75),
        "m_76_90": second_half & minute.ge(75),
        "m_1_30": minute.lt(30),
        "m_16_45": first_half & minute.ge(15),
        "m_31_60": (first_half & minute.ge(30)) | (second_half & minute.lt(60)),
        "m_46_75": second_half & minute.lt(75),
        "m_61_90": second_half & minute.ge(60),
    }



def add_possession_stats(team_stats: pd.DataFrame) -> pd.DataFrame:
    team_stats = team_stats.copy()
    if "pass_attempt" not in team_stats.columns:
        team_stats["possession"] = np.nan
        return team_stats

    pass_total = team_stats.groupby("game_id")["pass_attempt"].transform("sum")
    team_stats["possession"] = safe_div(team_stats["pass_attempt"], pass_total)
    return team_stats

def align_team_fouls(team_stats: pd.DataFrame) -> pd.DataFrame:
    required = {"game_id", "team_id", "opponent_team_id", "foul_committed", "foul_suffered"}
    if not required.issubset(team_stats.columns):
        return team_stats

    team_stats = team_stats.copy()
    committed_lookup = team_stats.set_index(["game_id", "team_id"])["foul_committed"]
    opponent_keys = pd.MultiIndex.from_frame(team_stats[["game_id", "opponent_team_id"]])
    team_stats["foul_suffered"] = committed_lookup.reindex(opponent_keys).fillna(0).to_numpy()
    return team_stats

def summarize_by_team(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    game_events, derived_columns = add_derived_event_stats(game_events)
    effective_stat_columns = [c for c in [*stat_columns, *derived_columns] if c in game_events.columns]
    game_events[effective_stat_columns] = game_events[effective_stat_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
    team_stats = (
        game_events.groupby(GAME_TEAM_FEATURES, dropna=False)[effective_stat_columns]
        .sum()
        .reset_index()
    )
    team_stats = align_team_fouls(team_stats)
    team_stats = add_rate_columns(team_stats)
    return add_possession_stats(team_stats)

def stat_dict(row: pd.Series, stat_value_columns: list[str]) -> dict[str, Any]:
    out = {}
    for col in stat_value_columns:
        value = row.get(col)
        if isinstance(value, np.generic):
            value = value.item()
        out[col] = value
    return out

def add_score_stats(interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]], game_events: pd.DataFrame) -> None:
    team_pairs = game_events[["game_id", "team_id", "opponent_team_id"]].drop_duplicates()
    for _, team_row in team_pairs.iterrows():
        key = (int(team_row["game_id"]), int(team_row["team_id"]))
        opponent_key = (int(team_row["game_id"]), int(team_row["opponent_team_id"]))
        for interval, stats in interval_lookup.get(key, {}).items():
            opponent_stats = interval_lookup.get(opponent_key, {}).get(interval, {})
            goals = int(stats.get("shot_goal", 0) or 0) + int(opponent_stats.get("shot_own_goal", 0) or 0)
            possession = stats.pop("possession", None)
            stats.pop("goals_for", None)
            reordered_stats = {}
            if possession is not None:
                reordered_stats["possession"] = possession
            reordered_stats["goals"] = goals
            reordered_stats.update(stats)
            stats.clear()
            stats.update(reordered_stats)

def build_interval_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> dict[tuple[int, int], dict[str, dict[str, Any]]]:

    interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]] = {}
    for interval, mask in interval_masks(game_events).items():
        interval_events = game_events.loc[mask].copy()
        if interval_events.empty:
            continue
        interval_stats = summarize_by_team(interval_events, stat_columns)
        stat_value_columns = [c for c in interval_stats.columns if c not in GAME_TEAM_FEATURES]
        for _, row in interval_stats.iterrows():
            key = (int(row["game_id"]), int(row["team_id"]))
            interval_lookup.setdefault(key, {})[interval] = stat_dict(row, stat_value_columns)
    add_score_stats(interval_lookup, game_events)
    return interval_lookup

def add_opponent_full_time_stats(team_stats: pd.DataFrame) -> pd.DataFrame:
    nested_columns = {"stats_by_interval", "opp_stats_by_interval"}
    value_columns = [c for c in team_stats.columns if c not in GAME_TEAM_FEATURES and c not in nested_columns]
    opponent_stats = team_stats[["game_id", "team_id", *value_columns]].rename(
        columns={"team_id": "opponent_team_id", **{col: f"opp_{col}" for col in value_columns}}
    )
    return team_stats.merge(opponent_stats, on=["game_id", "opponent_team_id"], how="left")

def build_game_team_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    feature_columns = [c for c in GAME_TEAM_FEATURES if c in game_events.columns]
    if not feature_columns:
        raise ValueError("No valid team match feature columns found in config game_stats.team_match_features.")

    team_stats = game_events[feature_columns].drop_duplicates().reset_index(drop=True).copy()
    interval_lookup = build_interval_stats(game_events, stat_columns)

    team_stats["stats"] = team_stats.apply(
        lambda row: interval_lookup.get((int(row["game_id"]), int(row["team_id"])), {}),
        axis=1,
    )
    team_stats["opp_stats"] = team_stats.apply(
        lambda row: interval_lookup.get((int(row["game_id"]), int(row["opponent_team_id"])), {}),
        axis=1,
    )
    return team_stats[feature_columns + ["stats", "opp_stats"]]

game_team_stats = build_game_team_stats(sample_game_events, stat_columns)
print(game_team_stats.shape)
game_team_stats.head()


(2, 14)


,game_id,game_date,game_status,competition_name,competition_country,season,week,team_id,team_name,opponent_team_id,opponent_team_name,game_venue,stats,opp_stats
0,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,796,Union Berlin,7614,RB Leipzig,away,"{'ft': {'possession': 0.3333333333333333, 'goa...","{'ft': {'possession': 0.6666666666666666, 'goa..."
1,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,7614,RB Leipzig,796,Union Berlin,home,"{'ft': {'possession': 0.6666666666666666, 'goa...","{'ft': {'possession': 0.3333333333333333, 'goa..."


In [24]:
game_team_stats.iloc[1]['stats']['ft']

{'possession': 0.6666666666666666,
 'goals': 3,
 'shot_event': 20,
 'shot_goal': 3,
 'shot_on_target': 10,
 'shot_off_target': 6,
 'shot_woodwork': 1,
 'shot_blocked': 4,
 'shot_own_goal': 0,
 'shot_zone_6_yard_box': 2,
 'shot_zone_penalty_area': 12,
 'shot_zone_outside_box': 6,
 'shot_open_play': 13,
 'shot_fastbreak': 2,
 'shot_set_piece': 5,
 'shot_penalty': 0,
 'shot_right_foot': 11,
 'shot_left_foot': 7,
 'shot_head': 2,
 'shot_other_body_part': 0,
 'pass_attempt': 642,
 'pass_completed': 546,
 'pass_cross': 19,
 'pass_freekick': 11,
 'pass_corner': 10,
 'pass_through_ball': 3,
 'pass_throw_in': 17,
 'pass_key_pass': 14,
 'pass_long': 49,
 'pass_short': 593,
 'pass_chipped': 42,
 'pass_ground': 600,
 'pass_head': 19,
 'pass_feet': 623,
 'pass_forward': 377,
 'pass_backward': 261,
 'pass_left': 316,
 'pass_right': 323,
 'pass_defensive_third': 124,
 'pass_mid_third': 363,
 'pass_final_third': 155,
 'dribble_event': 25,
 'dribble_successful': 13,
 'dribble_unsuccessful': 12,
 'tackl

## Build Game Player Stats

In [25]:
PLAYER_MATCH_FEATURES = config.get("game_stats", {}).get("player_match_features", [])


def is_per_90_source_stat(stat_name: str, value: Any) -> bool:
    return True


def add_player_per_90_stats(stats: dict[str, dict[str, Any]], minutes_played: Any) -> dict[str, dict[str, Any]]:
    if "ft" not in stats:
        return stats

    minutes = pd.to_numeric(pd.Series([minutes_played]), errors="coerce").iloc[0]
    minutes = None if pd.isna(minutes) else float(minutes)
    per_90_stats: dict[str, Any] = {}
    for source_col, raw_value in stats["ft"].items():
        if not is_per_90_source_stat(source_col, raw_value):
            continue
        value = pd.to_numeric(pd.Series([raw_value]), errors="coerce").iloc[0]
        if pd.isna(value) or minutes is None or minutes == 0:
            per_90_stats[source_col] = None
        else:
            per_90_stats[source_col] = float(value) / minutes * 90

    stats["per_90"] = per_90_stats
    return stats


def event_match_seconds(game_events: pd.DataFrame) -> pd.Series:
    minute = pd.to_numeric(game_events["minute"], errors="coerce").fillna(0)
    second = pd.to_numeric(game_events.get("second", 0), errors="coerce").fillna(0)
    return minute.mul(60).add(second)


def add_player_appearance_columns(game_events: pd.DataFrame) -> pd.DataFrame:
    game_events = game_events.copy()
    if game_events.empty or not {"game_id", "team_id", "player_id"}.issubset(game_events.columns):
        return game_events

    event_seconds = event_match_seconds(game_events)
    match_end_seconds = event_seconds.groupby(game_events["game_id"]).transform("max")
    player_key = ["game_id", "team_id", "player_id"]

    sub_on_seconds = event_seconds.where(game_events["type"].eq("SubstitutionOn")).groupby(
        [game_events[c] for c in player_key], dropna=False
    ).min()
    sub_off_seconds = event_seconds.where(game_events["type"].eq("SubstitutionOff")).groupby(
        [game_events[c] for c in player_key], dropna=False
    ).min()

    appearance = game_events[player_key].dropna().drop_duplicates().copy()
    appearance = appearance.merge(
        sub_on_seconds.rename("sub_on_seconds").reset_index(),
        on=player_key,
        how="left",
    )
    appearance = appearance.merge(
        sub_off_seconds.rename("sub_off_seconds").reset_index(),
        on=player_key,
        how="left",
    )
    appearance["starting_lineup"] = appearance["sub_on_seconds"].isna()

    match_end_by_game = game_events.assign(_match_end_seconds=match_end_seconds)[["game_id", "_match_end_seconds"]].drop_duplicates()
    appearance = appearance.merge(match_end_by_game, on="game_id", how="left")
    appearance["start_seconds"] = appearance["sub_on_seconds"].fillna(0)
    appearance["end_seconds"] = appearance["sub_off_seconds"].fillna(appearance["_match_end_seconds"])
    appearance["minutes_played"] = (
        (appearance["end_seconds"] - appearance["start_seconds"]).clip(lower=0).div(60).apply(np.ceil).astype(int)
    )

    appearance = appearance[player_key + ["starting_lineup", "minutes_played"]]
    game_events = game_events.merge(appearance, on=player_key, how="left")
    game_events["starting_lineup"] = game_events["starting_lineup"].fillna(False).astype(bool)
    game_events["minutes_played"] = pd.to_numeric(game_events["minutes_played"], errors="coerce").astype("Int64")
    return game_events


def summarize_by_player(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    player_events = game_events[game_events["player_id"].notna()].copy()
    if player_events.empty:
        return pd.DataFrame()

    player_events, derived_columns = add_derived_event_stats(player_events)
    effective_stat_columns = [c for c in [*stat_columns, *derived_columns] if c in player_events.columns]
    feature_columns = [c for c in PLAYER_MATCH_FEATURES if c in player_events.columns]
    if not feature_columns:
        raise ValueError("No valid player match feature columns found in config game_stats.player_match_features.")
    if not effective_stat_columns:
        return player_events[feature_columns].drop_duplicates().reset_index(drop=True)

    player_events[effective_stat_columns] = (
        player_events[effective_stat_columns]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
    )
    player_stats = (
        player_events.groupby(feature_columns, dropna=False)[effective_stat_columns]
        .sum()
        .reset_index()
    )
    player_stats = add_rate_columns(player_stats)

    return player_stats


def build_player_interval_stats(
    game_events: pd.DataFrame,
    stat_columns: list[str],
) -> dict[tuple[int, int], dict[str, dict[str, Any]]]:
    interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]] = {}
    for interval, mask in interval_masks(game_events).items():
        interval_events = game_events.loc[mask].copy()
        if interval_events.empty:
            continue

        interval_stats = summarize_by_player(interval_events, stat_columns)
        if interval_stats.empty:
            continue

        feature_columns = [c for c in PLAYER_MATCH_FEATURES if c in interval_stats.columns]
        stat_value_columns = [c for c in interval_stats.columns if c not in feature_columns]
        for _, row in interval_stats.iterrows():
            key = (int(row["game_id"]), int(row["player_id"]))
            interval_lookup.setdefault(key, {})[interval] = stat_dict(row, stat_value_columns)

    return interval_lookup


def build_game_player_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    player_events = add_player_appearance_columns(game_events)
    player_events = player_events[player_events["player_id"].notna()].copy()
    feature_columns = [c for c in PLAYER_MATCH_FEATURES if c in player_events.columns]
    if player_events.empty:
        return pd.DataFrame(columns=feature_columns + ["stats"])
    if not feature_columns:
        raise ValueError("No valid player match feature columns found in config game_stats.player_match_features.")

    players = (
        player_events[feature_columns]
        .drop_duplicates()
        .reset_index(drop=True)
        .copy()
    )
    interval_lookup = build_player_interval_stats(player_events, stat_columns)
    players["stats"] = players.apply(
        lambda row: interval_lookup.get((int(row["game_id"]), int(row["player_id"])), {}),
        axis=1,
    )
    players["stats"] = players.apply(
        lambda row: add_player_per_90_stats(row["stats"], row.get("minutes_played")),
        axis=1,
    )
    return players[feature_columns + ["stats"]]


game_events = prepare_game_events(load_game_events(GAME_ID))
game_stat_columns = [c for c in stat_columns if c in game_events.columns]
game_player_stats = build_game_player_stats(game_events, game_stat_columns)
print(game_player_stats.shape)
game_player_stats.head()


(32, 17)


,game_id,game_date,game_status,competition_name,competition_country,season,week,team_id,team_name,opponent_team_id,opponent_team_name,game_venue,player_id,player,starting_lineup,minutes_played,stats
0,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,796,Union Berlin,7614,RB Leipzig,away,369687.0,András Schäfer,True,83,"{'ft': {'shot_event': 0, 'shot_goal': 0, 'shot..."
1,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,796,Union Berlin,7614,RB Leipzig,away,303076.0,Danilho Doekhi,True,94,"{'ft': {'shot_event': 2, 'shot_goal': 1, 'shot..."
2,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,796,Union Berlin,7614,RB Leipzig,away,426239.0,Andrej Ilic,True,59,"{'ft': {'shot_event': 2, 'shot_goal': 0, 'shot..."
3,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,7614,RB Leipzig,796,Union Berlin,home,104917.0,Willi Orbán,True,94,"{'ft': {'shot_event': 0, 'shot_goal': 0, 'shot..."
4,1910873,2026-04-24 19:30:00,finished,Bundesliga,Germany,2025-2026,31,7614,RB Leipzig,796,Union Berlin,home,365332.0,Maarten Vandevoordt,True,94,"{'ft': {'shot_event': 0, 'shot_goal': 0, 'shot..."


In [26]:
game_player_stats.iloc[0].stats.keys()

dict_keys(['ft', 'fh', 'sh', 'm_1_15', 'm_16_30', 'm_31_45', 'm_46_60', 'm_61_75', 'm_76_90', 'm_1_30', 'm_16_45', 'm_31_60', 'm_46_75', 'm_61_90', 'per_90'])

In [27]:
game_player_stats.iloc[0].stats.keys()

dict_keys(['ft', 'fh', 'sh', 'm_1_15', 'm_16_30', 'm_31_45', 'm_46_60', 'm_61_75', 'm_76_90', 'm_1_30', 'm_16_45', 'm_31_60', 'm_46_75', 'm_61_90', 'per_90'])

In [28]:
game_player_stats.iloc[0]

game_id                                                          1910873
game_date                                            2026-04-24 19:30:00
game_status                                                     finished
competition_name                                              Bundesliga
competition_country                                              Germany
season                                                         2025-2026
week                                                                  31
team_id                                                              796
team_name                                                   Union Berlin
opponent_team_id                                                    7614
opponent_team_name                                            RB Leipzig
game_venue                                                          away
player_id                                                       369687.0
player                                             

In [29]:
game_player_stats.iloc[0].stats['ft']

{'shot_event': 0,
 'shot_goal': 0,
 'shot_on_target': 0,
 'shot_off_target': 0,
 'shot_woodwork': 0,
 'shot_blocked': 0,
 'shot_own_goal': 0,
 'shot_zone_6_yard_box': 0,
 'shot_zone_penalty_area': 0,
 'shot_zone_outside_box': 0,
 'shot_open_play': 0,
 'shot_fastbreak': 0,
 'shot_set_piece': 0,
 'shot_penalty': 0,
 'shot_right_foot': 0,
 'shot_left_foot': 0,
 'shot_head': 0,
 'shot_other_body_part': 0,
 'pass_attempt': 32,
 'pass_completed': 23,
 'pass_cross': 0,
 'pass_freekick': 0,
 'pass_corner': 0,
 'pass_through_ball': 0,
 'pass_throw_in': 0,
 'pass_key_pass': 0,
 'pass_long': 6,
 'pass_short': 26,
 'pass_chipped': 5,
 'pass_ground': 27,
 'pass_head': 2,
 'pass_feet': 30,
 'pass_forward': 17,
 'pass_backward': 15,
 'pass_left': 17,
 'pass_right': 14,
 'pass_defensive_third': 8,
 'pass_mid_third': 15,
 'pass_final_third': 9,
 'dribble_event': 0,
 'dribble_successful': 0,
 'dribble_unsuccessful': 0,
 'tackle_attempted_event': 2,
 'tackle_gained_possession': 0,
 'tackle_did_not_get_po

In [30]:
game_player_stats.iloc[0].stats['per_90']

{'shot_event': 0.0,
 'shot_goal': 0.0,
 'shot_on_target': 0.0,
 'shot_off_target': 0.0,
 'shot_woodwork': 0.0,
 'shot_blocked': 0.0,
 'shot_own_goal': 0.0,
 'shot_zone_6_yard_box': 0.0,
 'shot_zone_penalty_area': 0.0,
 'shot_zone_outside_box': 0.0,
 'shot_open_play': 0.0,
 'shot_fastbreak': 0.0,
 'shot_set_piece': 0.0,
 'shot_penalty': 0.0,
 'shot_right_foot': 0.0,
 'shot_left_foot': 0.0,
 'shot_head': 0.0,
 'shot_other_body_part': 0.0,
 'pass_attempt': 34.6987951807229,
 'pass_completed': 24.93975903614458,
 'pass_cross': 0.0,
 'pass_freekick': 0.0,
 'pass_corner': 0.0,
 'pass_through_ball': 0.0,
 'pass_throw_in': 0.0,
 'pass_key_pass': 0.0,
 'pass_long': 6.506024096385542,
 'pass_short': 28.192771084337352,
 'pass_chipped': 5.421686746987952,
 'pass_ground': 29.277108433734938,
 'pass_head': 2.168674698795181,
 'pass_feet': 32.53012048192771,
 'pass_forward': 18.433734939759034,
 'pass_backward': 16.265060240963855,
 'pass_left': 18.433734939759034,
 'pass_right': 15.180722891566264,